In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto", 
    device_map="auto"
)

In [ ]:
def generate_text(prompt, max_new_tokens=100):
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Updated to use llm_tokenizer and llm_model
    model_inputs = llm_tokenizer([text], return_tensors="pt").to(llm_model.device)

    generated_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False 
    )
    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    return llm_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

In [ ]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter


In [ ]:
path = "/kaggle/input/datasets/mazenbassem/tips-hindawi-university-info/Tips Hindawi University Info.pdf"
loader = PyPDFLoader(path)
document = loader.load()

text_splitter = CharacterTextSplitter(chunk_size = 100, chunk_overlap= 50)
chunks = text_splitter.split_documents(document)

embedding_model_name='sentence-transformers/all-MiniLM-L6-v2'
embedding = HuggingFaceEmbeddings(model_name = embedding_model_name)

vectorDB = FAISS.from_documents(chunks, embedding)

In [ ]:
def ask_question(query):
    # 1. Similarity search using your LangChain Vector Store
    docs = vectorDB.similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Strict prompt template to maintain your THU accuracy guidelines
    prompt = f"""You are a strict, factual assistant. Answer the Question using ONLY the facts explicitly stated in the Background Context below.
Do not assume, extrapolate, or infer any information not directly mentioned. If the context does not contain the answer, reply exactly with: "The provided context does not contain this information."

Background Context:
{context}

Question: {query}
Answer:"""
    
    # FIX 1: Change max_length=1000 to max_new_tokens=150
    result_list = generate_text(prompt, max_new_tokens=150)
    
    # FIX 2: Extract the string from the list before treating it like a string
    return result_list[0].strip()


if __name__ == "__main__":
    print("PDF Q&A System with Mistral / Qwen \n")
    while True:
        user_query = input("Ask a question (or type 'exit'): ")
        if user_query.lower() == "exit":
            break
            
        answer = ask_question(user_query)
        
        # Since our prompt template cuts off right at "Answer:", 
        # and generate_text slices away the prompt, `answer` is already just the response!
        print(f"\nAnswer: {answer}\n")
        print("-" * 50)